# Notebook 07 — Script-Based CKA Decomposition: Novel Technique 4

**TIN-7: Cross-Lingual Embedding Alignment Analysis**

This notebook tests a critical hypothesis: is cross-lingual
alignment driven by **token-surface similarity** (languages sharing
the same script have similar BPE tokens, leading to higher CKA) or
**true semantic alignment** (the model understands meaning regardless
of script)?

## Script groups in our 13 languages

| Script      | Languages                              |
|-------------|----------------------------------------|
| Latin       | en, es, fr, de, sw, tr, yo (7 langs)   |
| Arabic      | ar, fa (2 langs)                       |
| Devanagari  | hi (1 lang)                            |
| Bengali     | bn (1 lang)                            |
| Tamil       | ta (1 lang)                            |
| Ge'ez       | am (1 lang)                            |

In [ ]:
import logging
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.languages import Language, SCRIPT_GROUPS
from src.analysis.cross_lingual_embedding_alignment.clustering import compute_script_group_metrics
from src.analysis.cross_lingual_embedding_alignment.visualization import plot_script_decomposition

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

RESULTS_DIR = PROJECT_ROOT / "analysis" / "results" / "cross_lingual"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
CKA_DIR = RESULTS_DIR / "cka_matrices"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load CKA Matrices

In [ ]:
languages = list(Language)
language_names = [lang.lang_name for lang in languages]
iso_names = [lang.iso_code for lang in languages]
n_layers = 4

cka_matrices: dict[int, np.ndarray] = {}
for layer_idx in range(n_layers):
    filepath = CKA_DIR / f"layer_{layer_idx}_linear_cka.npy"
    if filepath.exists():
        cka_matrices[layer_idx] = np.load(filepath)

print(f"Loaded CKA matrices for {len(cka_matrices)} layers.")

## 2. Script Group Overview

In [ ]:
print("=== Script Groups ===")
for script, langs in sorted(SCRIPT_GROUPS.items()):
    lang_list = ", ".join(f"{l.iso_code} ({l.lang_name})" for l in langs)
    print(f"  {script:<12} ({len(langs)} langs): {lang_list}")

## 3. Script Decomposition Metrics per Layer

In [ ]:
script_results: dict[int, dict] = {}

print("\n=== Script Decomposition ===")
print(f"{'Layer':>5} {'Intra-Script':>14} {'Inter-Script':>14} {'Gap':>8}")
print("-" * 50)

for layer_idx, matrix in sorted(cka_matrices.items()):
    metrics = compute_script_group_metrics(
        similarity_matrix=matrix,
        language_names=language_names,
        languages=languages,
    )
    script_results[layer_idx] = metrics

    print(
        f"  {layer_idx:>3} {metrics['intra_script_cka']:>14.4f} "
        f"{metrics['inter_script_cka']:>14.4f} "
        f"{metrics['script_gap']:>8.4f}"
    )

## 4. Intra- vs Inter-Script CKA Convergence

In [ ]:
layer_indices = sorted(script_results.keys())
intra_values = [script_results[l]["intra_script_cka"] for l in layer_indices]
inter_values = [script_results[l]["inter_script_cka"] for l in layer_indices]

fig = plot_script_decomposition(
    layer_indices=list(layer_indices),
    intra_script_cka=intra_values,
    inter_script_cka=inter_values,
    save_path=str(FIGURES_DIR / "script_decomposition.png"),
)
plt.show()

## 5. Per-Script Average CKA

In [ ]:
print("\n=== Per-Script Intra-CKA by Layer ===")
for layer_idx in layer_indices:
    per_script = script_results[layer_idx]["per_script_avg_cka"]
    print(f"\nLayer {layer_idx}:")
    for script, avg_cka in sorted(per_script.items()):
        n_langs = len(SCRIPT_GROUPS.get(script, []))
        if n_langs >= 2:
            print(f"  {script:<12} avg_CKA={avg_cka:.4f} ({n_langs} languages)")
        else:
            print(f"  {script:<12} (only 1 lang — no intra-script pairs)")

## 6. Script Gap Over Layers (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

gaps = [script_results[l]["script_gap"] for l in layer_indices]
colors = ["#E91E63" if g > 0.05 else "#4CAF50" for g in gaps]

bars = ax.bar(
    [f"Layer {l}" for l in layer_indices],
    gaps,
    color=colors,
    edgecolor="white",
    linewidth=1,
)

for bar, gap in zip(bars, gaps):
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.005,
        f"{gap:.4f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

ax.axhline(y=0, color="gray", linestyle="-", linewidth=0.5)
ax.set_ylabel("Script Gap (Intra - Inter)", fontsize=12)
ax.set_title(
    "Script Bias Gap per Layer\n(Positive = script-dependent alignment)",
    fontsize=13, fontweight="bold",
)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "script_gap_bars.png", bbox_inches="tight")
plt.show()

## 7. Latin-Script Intra-CKA Deep Dive

With 7 Latin-script languages, we have enough pairs for detailed
analysis. Are Western European languages (en, es, fr, de) more
similar to each other than to African Latin-script languages (sw, yo)?

In [ ]:
latin_langs = [l for l in languages if l.script == "Latin"]
latin_indices = [language_names.index(l.lang_name) for l in latin_langs]
latin_iso = [l.iso_code for l in latin_langs]

for layer_idx in sorted(cka_matrices.keys()):
    full_matrix = cka_matrices[layer_idx]
    latin_matrix = full_matrix[np.ix_(latin_indices, latin_indices)]

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        latin_matrix,
        xticklabels=latin_iso,
        yticklabels=latin_iso,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        vmin=0.0,
        vmax=1.0,
        square=True,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title(
        f"Latin-Script Languages CKA — Layer {layer_idx}",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"latin_cka_layer_{layer_idx}.png", bbox_inches="tight")
    plt.show()

## 8. Save Metrics

In [ ]:
script_metrics: dict[str, float] = {}
for layer_idx, metrics in script_results.items():
    prefix = f"layer_{layer_idx}"
    script_metrics[f"{prefix}_intra_script_cka"] = metrics["intra_script_cka"]
    script_metrics[f"{prefix}_inter_script_cka"] = metrics["inter_script_cka"]
    script_metrics[f"{prefix}_script_gap"] = metrics["script_gap"]

with open(METRICS_DIR / "script_metrics.json", "w") as f:
    json.dump(script_metrics, f, indent=2)

print(f"Script metrics saved to {METRICS_DIR / 'script_metrics.json'}")

## 9. Summary

**Key findings:**

### Script decomposition results
- **The script gap is negative at every layer** (−0.070 to −0.073), meaning inter-script CKA
  (∼0.66–0.67) consistently exceeds intra-script CKA (∼0.59–0.60). This is the opposite of
  what a token-surface-similarity hypothesis would predict — languages sharing the same script
  are actually *less* similar in representation space than languages using different scripts.
- **The script gap remains stable across all four layers**, showing no convergence trend.
  This means the pattern is not an early-layer artifact that dissolves with depth — it persists
  throughout the entire network.
- **Per-script intra-CKA reveals a striking contrast.** Arabic-script languages (Arabic, Persian)
  have the highest intra-script CKA (∼0.72 across all layers), while the Latin-script group
  (7 languages) has much lower intra-script CKA (∼0.58–0.60). This suggests that the Latin
  group is internally diverse despite sharing a writing system.

### Cross-notebook synthesis
- **Script does not explain alignment.** Combined with Notebook 04's finding that language
  family also does not predict CKA clusters (ARI = −0.234), these results show that neither
  genetic family nor writing system organizes Tiny Aya's representations. The model's internal
  similarity structure is driven by other factors.
- **Anisotropy masks the true picture.** Notebook 05 showed that after whitening, all language
  pairs converge to CKA ≈ 1.0 from Layer 1 onward. The script-based differences observed here
  (standard CKA ∼0.59 vs ∼0.67) are therefore likely artifacts of the anisotropic geometry
  rather than genuine representational differences.
- **Geometric similarity ≠ functional alignment.** Notebook 06 demonstrated that retrieval
  performance is strongly stratified by script (Latin-script languages: MRR 0.43–0.49;
  non-Latin: MRR 0.009–0.30 at Layer 3), even though CKA shows inter-script pairs as *more*
  similar. This apparent paradox — high geometric similarity but low functional alignment —
  reinforces that aggregate CKA (even standard) cannot be used as a proxy for sentence-level
  retrieval quality.
- **Overall interpretation:** Tiny Aya learns a shared representational geometry across all 13
  languages (especially visible after whitening), but this shared geometry does not translate
  into fine-grained, sentence-level cross-lingual alignment for languages with different scripts
  or lower training resources. The model's cross-lingual capacity appears strongest within the
  Latin-script, high-resource cluster and weakest for Indic languages (Hindi, Bengali, Tamil).